In [ ]:
import pandapower as pp
import torch
import numpy as np
import julia
julia.install()

from julia.api import Julia
jl = Julia(compiled_modules=False)

net = pp.from_pickle("/home/iboero/Tesis/crear_modelo_uy/GNN4OPF/modelos_pp/uy_pp_net_v14_(sin_eolico_ni_solar).p")
net.bus["pm_param/setpoint_v"] = 1.0

X_test = np.load(f'/home/iboero/Tesis/unsupervised_uru/GNN4OPF/data/reduru/test/input.npy')


In [35]:
bus_map = {net.bus.index[i]: i for i in range(len(net.bus))}

# That also works for a list of buses
def bus_pos(buses):
    try:
        return [bus_map[bus] for bus in buses]
    except:
        return bus_map[buses]

In [ ]:
idx_gens = bus_pos(net.gen.bus.values.astype(int))
idx_load = bus_pos(net.load.bus.values.astype(int))
idx_sgens = bus_pos(net.sgen.bus.loc[net.sgen.controllable==False].values.astype(int))

idx = 0
net.load.loc[:,'p_mw'] = X_test[idx,idx_load,0] 
net.load.loc[:,'q_mvar'] = X_test[idx,idx_load,1]
net.gen.loc[:,'p_mw'] =  X_test[idx,idx_gens,2]
net.sgen[net.sgen['controllable'] == False].loc[:,'p_mw'] = X_test[idx,idx_sgens,3]

In [37]:
pp.runpm_vstab(net)
normal_net_volt = net.res_gen['vm_pu']

no costs are given - overall generated power is minimized



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



In [47]:
# sacamos una linea
net.line.drop(6, inplace=True)
pp.runpm_vstab(net)
altered_net_volt = net.res_gen['vm_pu']

no costs are given - overall generated power is minimized


In [48]:
np.abs(normal_net_volt - altered_net_volt)

0     0.000004
1     0.000119
2     0.001141
7     0.000932
12    0.000706
13    0.000034
14    0.000359
15    0.000388
18    0.000164
23    0.000775
24    0.000007
27    0.000310
Name: vm_pu, dtype: float64